# Output

**What this shows:** how to choose what XBeach writes and when: mean and instantaneous
maps, point timeseries, run-up gauges, output timing and hotstart files.

**Prerequisites:** [Tutorial 5: Choosing model settings](../tutorial/05_model_settings.ipynb).

**You will learn:**

- the four kinds of XBeach output and their timing parameters
- how to place output points and run-up gauges using grid coordinates
- how to write output at irregular times from a file
- which checks rompy-xbeach applies to output settings

**Data used:** none.

## Setup

In [1]:
import shutil
from pathlib import Path

from pydantic import ValidationError
from rompy.logging import config as logging_config
from rompy_xbeach.components.output import Output
from rompy_xbeach.grid import RegularGrid
from rompy_xbeach.types import XBeachDataBlob

logging_config.update(level="WARNING")  # rompy logs every step, show warnings only

OUT_DIR = Path("_output") / "output"
shutil.rmtree(OUT_DIR, ignore_errors=True)
OUT_DIR.mkdir(parents=True)

grid = RegularGrid(
    ori={"x": 115.594239, "y": -32.641104, "crs": 4326},
    alfa=347.0,
    dx=10.0,
    dy=15.0,
    nx=230,
    ny=220,
    crs=28350,
)


def show(component):
    """Print the XBeach parameters a component writes, lists one item per line."""
    for key, value in component.get(destdir=OUT_DIR).items():
        if isinstance(value, list):
            print("\n".join(str(item) for item in value))
        else:
            print(f"{key} = {value}")

## 1. Output types

| Type | Field | Interval | Written |
|---|---|---|---|
| Mean maps | `meanvars` | `tintm` | Mean, variance, min and max over each interval |
| Instantaneous maps | `globalvars` | `tintg` | Snapshots of the whole domain |
| Point timeseries | `pointvars` + `points` | `tintp` | Values at fixed locations |
| Run-up gauges | `pointvars` + `rugauges` | `tintp` | Values at the moving waterline |

All output starts at `tstart` (seconds from the start of the run). Output goes to a
NetCDF file by default.

In [2]:
show(
    Output(
        ncfilename="storm.nc",
        meanvars=["H", "zs", "u", "v"],
        tintm=3600.0,
        globalvars=["zb", "zs"],
        tintg=600.0,
        tstart=1800.0,
    )
)

outputformat = netcdf
ncfilename = storm.nc
tstart = 1800.0
tintg = 600.0
tintm = 3600.0
nmeanvar = 4
H
zs
u
v
nglobalvar = 2
zb
zs


## 2. Points and run-up gauges

Locations are given in the grid coordinate system (here MGA zone 50) and XBeach uses
the nearest grid point. Taking them from the grid arrays avoids mistakes. Here there
are three points along a cross-shore transect and a run-up gauge on the same
transect.

In [3]:
iy = grid.ny // 2
points = [(float(grid.x[iy, ix]), float(grid.y[iy, ix])) for ix in (50, 150, 200)]
rugauges = [points[0]]

show(Output(points=points, rugauges=rugauges, pointvars=["zs", "H", "u"], tintp=1.0))

outputformat = netcdf
tintp = 1.0
npoints = 3
369003.35662210465 6389121.824346607
369977.7266868899 6388896.873292263
370464.9117192825 6388784.397765091
nrugauge = 1
369003.35662210465 6389121.824346607
npointvar = 3
zs
H
u


## 3. Output at irregular times

Instead of a fixed interval, output times can come from a file: `tsglobal`, `tsmean`
or `tspoint`. The file has the number of times on the first line, then one time per
line. It is copied into the workspace, and times from a file take precedence over
intervals.

In [4]:
times_file = OUT_DIR / "source_global_times.txt"
times = [0, 600, 1200, 3600, 7200]
times_file.write_text(f"{len(times)}\n" + "\n".join(str(t) for t in times) + "\n")

show(Output(globalvars=["zb"], tsglobal=XBeachDataBlob(source=times_file)))

outputformat = netcdf
tsglobal = source_global_times.txt
nglobalvar = 1
zb


## 4. Hotstart files

`writehotstart` saves the model state so that another run can continue from it,
every `tinth` seconds or at the end of the run. See
[Hotstart and chained runs](hotstart_and_chained_runs.ipynb).

In [5]:
show(Output(writehotstart=True, tinth=3600.0))

outputformat = netcdf
tinth = 3600.0
writehotstart = True


## 5. Checks

Variable names are checked against the variables XBeach knows:

In [6]:
try:
    Output(globalvars=["Hs"])
except ValidationError as error:
    print(error.errors()[0]["msg"][:80], "...")

Input should be 'As', 'BR', 'Cdrag', 'D', 'D15', 'D50', 'D50top', 'D90', 'D90top ...


Inconsistent settings produce warnings, for example point variables without any
locations:

In [7]:
output = Output(pointvars=["zs"])

2026-09-28 14:11:33 [WARNING] rompy_xbeach.components.output: ! Point output variables (pointvars) are defined, but no point locations (points) or runup gauge locations (rugauges) have been prescribed. Output will not be generated.


`OutputVarsEnum` lists every variable name XBeach accepts:

In [8]:
from rompy_xbeach.types import OutputVarsEnum

print(
    len(OutputVarsEnum),
    "variables, for example:",
    [v.value for v in OutputVarsEnum][:12],
)

280 variables, for example: ['As', 'BR', 'Cdrag', 'D', 'D15', 'D50', 'D50top', 'D90', 'D90top', 'DR', 'Dc', 'Df']


## Summary

- Choose map, point and run-up output with the variable lists, and their timing with
  `tintm`, `tintg` and `tintp` or with time files.
- Give point and gauge locations in grid coordinates.
- The variable counts XBeach needs are filled in for you.